## Embeddings and Embedding Models

## Prerequisite Understanding:
We are now going towards learning how RAG works.<br>
Basically, when we as users provide documents to an AI, how does an AI understand from which part of this big document should it provide relevent answers to us users.<br>
First let us understand the meaning of 'vector embeddings':<br>
vector -> a list of numbers <br>
embeddings -> the process of converting text into that list of numbers<br>
So, <br>
vector embeddings -> a list of numbers that represents the meaning of a piece of text<br>
.........................................................<br>
The problem we're solving:<br>
Suppose we have 148 chunks of text. When a user asks a question, how do we know which chunks are relevant to answer it?<br>
We can't send all 148 chunks to the AI, cause its too expensive and slow. We need to find the 2-3 most relevant ones.<br>
That's what embeddings and vector stores solve.<br>
.........................................................<br>
Here is how it works:
1. We load our DOC into the Document Object.
2. We Split it into chunks.
3. Each chunk(set of texts) is converted into numbers (embeddings) and stored in a database 'Chromadb'.
4. When a user asks a question such as "What is a chain in LangChain?"
5. That question gets converted to numbers too..
6. Chroma finds which chunks in it have the most similar numbers (similar vectors in it, is comparison to vectors of the given question)
7. Those chunks get sent to the AI
8. AI reads those specific chunks and answers the question

.........................................................<br>
In Easy words:
So embeddings is a list of numbers..
And each chunk(a particular set of texts) has an 'embedding' stored in Chromadb (where embedding is like a coordinate in the chroma database map). And when a new question is asked by the user, that's also turned into a list of numbers, which is taken in like an embedding to chromadb, and gives results related to other chunks lying around this new embedding.<br>
.........................................................<br>
Example of how an embedding looks like:<br>
(suppose we have 3 chunks:)<br>
"LangChain is a framework" → [0.23, -0.51, 0.87, 0.12, ...]<br>
"Python is a language"     → [0.19, -0.48, 0.91, 0.09, ...]<br>
"I love pizza"             → [-0.72, 0.33, -0.15, 0.61, ...]<br>
The key insight is that "similar meaning = similar numbers".<br>
The sentences "LangChain is a framework" and "Python is a language" are both about programming, so their vectors are close to each other.<br> "I love pizza" is completely different, so its vector is far away. <br>
[Don't see the numbers, we won't understand like that. Just imagine Chromadb like a x-y graph, where these embeddings are like 'coordinates'. Chunks/Texts having similar meaning lie near each other in the graph, or in ChromaDB.]<br>
ChromaDB -> It's a vector database designed to store and search vectors efficiently. Regular databases search by exact match. Chroma searches by similarity. Its like: "find me the closest vectors to this query vector."<br>
.........................................................<br>
PDF → chunks → embeddings → stored in Chroma<br>
User question → embedding → similarity search in Chroma → relevant chunks → AI → answer<br>
.........................................................<br>
Now, before moving ahead, let's understand how chunks are stored in ChromaDB.<br>
These chunks can be anything from the given doc: simple phrases, sentences, or even big paragraphs.<br>
Anyways, the whole chunk is converted into a single vector, not individual parts from the chunk.<br>
.........................................................<br>
What about Ambiguous words? <br>
Because if that's the case, a chunk can also be a big paragraph..sometimes big enough that a single 'word' can even have 2 meanings in itself.Something like: "I went to the bank to deposit money, then walked along the river bank to relax."<br>
Both meanings of "bank" are in the same chunk, so what does the vector look like?<br>
In such a situation, The vector becomes a blend of both meanings. It gets pulled in both directions. <br>
Yes, this can cause problems sometimes. This is a real limitation of embeddings called the "semantic blur" problem.<br>
If someone searches "river bank" this chunk might show up. If someone searches "financial bank" this chunk might also show up. It's not perfectly precise.<br>
How real RAG systems handle this:<br>
1. Smaller chunks — less chance of two conflicting meanings in one chunk<br>
2. Better embedding models — newer models handle context and ambiguity much better<br>
3. Reranking — after similarity search, a second model re-reads the actual text and reranks results by true relevance<br>

.........................................................<br>
Also, problems with the size of chunks can arrive:<br>
Small chunk: "LangChain builds AI apps"  → very specific vector, easy to match<br>
Big chunk: "LangChain builds AI apps. I love pizza. The weather is nice." → confused vector, harder to match accurately<br>
This is exactly why chunk size matters so much in RAG. If its too big, the embedding loses focus. And if its too small, it loses context.<br>
The sweet spot is usually chunks that cover one idea or topic, so not too broad, nor too narrow.<br>
.........................................................<br>

In [1]:
%%capture
!pip install --force-reinstall --no-cache-dir tenacity==8.2.3 --user
!pip install "ibm-watsonx-ai==1.0.8" --user
!pip install "ibm-watson-machine-learning==1.0.367" --user
!pip install "langchain-ibm==0.1.7" --user
!pip install "langchain-community==0.2.10" --user
!pip install "langchain-experimental==0.0.62" --user
!pip install "langchainhub==0.1.18" --user
!pip install "langchain==0.2.11" --user
!pip install "pypdf==4.2.0" --user
!pip install "chromadb==0.4.24" --user

In [ ]:
import os
os._exit(00)

In [1]:
# We can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
import os
os.environ['ANONYMIZED_TELEMETRY'] = 'False'

from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM

Loading the Doc into a Document Object:

In [2]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/96-FDF8f7coh0ooim7NyEQ/langchain-paper.pdf")
document = loader.load()

In [3]:
from langchain.text_splitter import CharacterTextSplitter
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20, separator="\n")
chunk1 = text_splitter.split_documents(document)

## Now, working upon Embeddings from here:

In [4]:
# Import the EmbedTextParamsMetaNames class from ibm_watsonx_ai.metanames module
# This class provides constants for configuring Watson embedding parameters
from ibm_watsonx_ai.metanames import EmbedTextParamsMetaNames

# Configure embedding parameters using a dictionary:
# - TRUNCATE_INPUT_TOKENS: Limit the input to 3 tokens (very short, possibly for testing)
# - RETURN_OPTIONS: Request that the original input text be returned along with embeddings
embed_params = {
 EmbedTextParamsMetaNames.TRUNCATE_INPUT_TOKENS: 3,
 EmbedTextParamsMetaNames.RETURN_OPTIONS: {"input_text": True},
}
#TRUNCATE_INPUT_TOKENS: 3  - if a chunk is too long, cut it to 3 tokens max before embedding. This is set very low for testing — in real usage we'd set it much higher like 512.
#RETURN_OPTIONS: {"input_text": True} — tells IBM to send back the original text along with the embedding. Useful for debugging — we can verify which text produced which vector.


In [5]:
# Import the WatsonxEmbeddings class from langchain_ibm module
# This provides an integration between LangChain and IBM's Watson AI services
from langchain_ibm import WatsonxEmbeddings

# Create a WatsonxEmbeddings instance with the following configuration:
# - model_id: Specifies the "slate-125m-english-rtrvr-v2" embedding model from IBM
# - url: The endpoint URL for the Watson service in the US South region
# - project_id: The Watson project ID to use ("skills-network")
# - params: The embedding parameters configured earlier
watsonx_embedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url="https://us-south.ml.cloud.ibm.com",
    project_id="skills-network",
    params=embed_params,
)

In [7]:
texts = [text.page_content for text in chunk1]
# This is a list comprehension — it loops through all 148 chunks and pulls out just the text content from each one
embedding_result = watsonx_embedding.embed_documents(texts) #This sends all 148 texts to IBM's embedding model and gets back 148 vectors
embedding_result[0][:5] # '0' is first chunk's vector | ':5' is just the first 5 numbers out of hundreds

[-0.04104941338300705,
 0.01380295492708683,
 -0.05267169326543808,
 0.011825969442725182,
 0.03604104742407799]